In [2]:
import subprocess
import sys

subprocess.run(
    [
        "uv", "pip", "install", "--python", sys.executable,
        "kagglehub", "matplotlib", "numpy", "pandas", "ipython",
        "scikit-learn", "torch", "seaborn",
    ],
    check=True,
)

CompletedProcess(args=['uv', 'pip', 'install', '--python', 'c:\\Users\\IvoAg\\AppData\\Local\\Python\\pythoncore-3.14-64\\python.exe', 'kagglehub', 'matplotlib', 'numpy', 'pandas', 'ipython', 'scikit-learn', 'torch', 'seaborn'], returncode=0)

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch import nn, optim

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier

In [4]:
from pathlib import Path
import kagglehub
import pandas as pd
from IPython.display import display

path = Path(kagglehub.dataset_download(
    "mohamedamineferrag/edgeiiotset-cyber-security-dataset-of-iot-iiot"
))
print("Dataset directory:", path)

csv_files = sorted(file for file in path.rglob("*") if file.suffix.lower() == ".csv")
if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {path}")

display(pd.DataFrame({
    "file": [str(file.relative_to(path)) for file in csv_files],
    "size_mb": [round(file.stat().st_size / 1024**2, 2) for file in csv_files],
}))

c:\Users\IvoAg\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset directory: C:\Users\IvoAg\.cache\kagglehub\datasets\mohamedamineferrag\edgeiiotset-cyber-security-dataset-of-iot-iiot\versions\5


,file,size_mb
0,Edge-IIoTset dataset\Attack traffic\Backdoor_a...,17.06
1,Edge-IIoTset dataset\Attack traffic\DDoS_HTTP_...,91.28
2,Edge-IIoTset dataset\Attack traffic\DDoS_ICMP_...,850.75
3,Edge-IIoTset dataset\Attack traffic\DDoS_TCP_S...,897.59
4,Edge-IIoTset dataset\Attack traffic\DDoS_UDP_F...,858.53
5,Edge-IIoTset dataset\Attack traffic\MITM_attac...,0.31
6,Edge-IIoTset dataset\Attack traffic\OS_Fingerp...,0.30
7,Edge-IIoTset dataset\Attack traffic\Password_a...,623.03
8,Edge-IIoTset dataset\Attack traffic\Port_Scann...,7.00
9,Edge-IIoTset dataset\Attack traffic\Ransomware...,7.24


In [10]:
csv_path = next(
    (
        file for file in csv_files
        if file.name.casefold() == "ml-edgeiiot-dataset.csv"
    ),
    None,
)
if csv_path is None:
    raise FileNotFoundError(f"ML-EdgeIIoT dataset não encontrado em {path}")

print("Loading the complete selected dataset...")
print("Selected CSV:", csv_path.relative_to(path))
df = pd.read_csv(csv_path, low_memory=False)
print(f"Loaded data: {df.shape[0]:,} rows and {df.shape[1]} columns")
display(df.head(10))

Loading the complete selected dataset...
Selected CSV: Edge-IIoTset dataset\Selected dataset for ML and DL\ML-EdgeIIoT-dataset.csv
Loaded data: 157,800 rows and 63 columns


,frame.time,ip.src_host,ip.dst_host,arp.dst.proto_ipv4,arp.opcode,arp.hw.size,arp.src.proto_ipv4,icmp.checksum,icmp.seq_le,icmp.transmit_timestamp,...,mqtt.proto_len,mqtt.protoname,mqtt.topic,mqtt.topic_len,mqtt.ver,mbtcp.len,mbtcp.trans_id,mbtcp.unit_id,Attack_label,Attack_type
0,6.0,192.168.0.152,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
1,6.0,192.168.0.101,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
2,6.0,192.168.0.152,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
3,6.0,192.168.0.101,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
4,6.0,192.168.0.152,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
5,6.0,192.168.0.101,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
6,6.0,192.168.0.152,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
7,6.0,192.168.0.101,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
8,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
9,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM


In [11]:
n_rows, n_columns = df.shape
print(f"Linhas: {n_rows:,}")
print(f"Colunas: {n_columns:,}")

print("\nTipos de dados:")
display(df.dtypes.value_counts().rename_axis("tipo").to_frame("quantidade"))

missing = df.isna().sum()
missing_summary = pd.DataFrame({
    "quantidade": missing,
    "percentual (%)": (100 * missing / len(df)).round(2),
})
missing_summary = missing_summary[missing_summary["quantidade"] > 0].sort_values(
    "quantidade", ascending=False
)

print("\nValores ausentes (NaN):")
if missing_summary.empty:
    print("Nenhum valor ausente encontrado.")
else:
    display(missing_summary)

numeric = df.select_dtypes(include=np.number)
infinite_counts = numeric.apply(lambda column: np.isinf(column).sum())
infinite_counts = infinite_counts[infinite_counts > 0]

print("\nValores infinitos:")
if infinite_counts.empty:
    print("Nenhum valor infinito encontrado.")
else:
    display(infinite_counts.rename("quantidade").to_frame())

print("\nEstatísticas das colunas numéricas:")
stats = numeric.apply(
    lambda column: column.replace([np.inf, -np.inf], np.nan).describe()
).T
display(stats[["count", "mean", "std", "min", "25%", "50%", "75%", "max"]])

categorical = df.select_dtypes(exclude=np.number)
if not categorical.empty:
    print("\nResumo das colunas categóricas:")
    display(categorical.describe().T)

for target in ("Attack_label", "Attack_type"):
    if target in df.columns:
        counts = df[target].value_counts(dropna=False)
        print(f"\nDistribuição de {target}:")
        display(pd.DataFrame({
            "quantidade": counts,
            "percentual (%)": (100 * counts / len(df)).round(2),
        }))

Linhas: 157,800
Colunas: 63

Tipos de dados:


,quantidade
tipo,
float64,42
str,20
int64,1



Valores ausentes (NaN):
Nenhum valor ausente encontrado.

Valores infinitos:
Nenhum valor infinito encontrado.

Estatísticas das colunas numéricas:


,count,mean,std,min,25%,50%,75%,max
arp.opcode,157800.0,1.419518e-02,1.497828e-01,0.0,0.0,0.000000e+00,0.000000e+00,2.000000e+00
arp.hw.size,157800.0,5.984791e-02,5.962449e-01,0.0,0.0,0.000000e+00,0.000000e+00,6.000000e+00
icmp.checksum,157800.0,3.047292e+03,1.114433e+04,0.0,0.0,0.000000e+00,0.000000e+00,6.553200e+04
icmp.seq_le,157800.0,3.239980e+03,1.140607e+04,0.0,0.0,0.000000e+00,0.000000e+00,6.552400e+04
icmp.transmit_timestamp,157800.0,4.046816e+04,1.764075e+06,0.0,0.0,0.000000e+00,0.000000e+00,7.728902e+07
icmp.unused,157800.0,0.000000e+00,0.000000e+00,0.0,0.0,0.000000e+00,0.000000e+00,0.000000e+00
http.content_length,157800.0,1.471552e+01,2.296597e+02,0.0,0.0,0.000000e+00,0.000000e+00,8.365500e+04
http.response,157800.0,4.574778e-02,2.089383e-01,0.0,0.0,0.000000e+00,0.000000e+00,1.000000e+00
http.tls_port,157800.0,0.000000e+00,0.000000e+00,0.0,0.0,0.000000e+00,0.000000e+00,0.000000e+00
tcp.ack,157800.0,7.160039e+07,3.101231e+08,0.0,0.0,1.000000e+00,4.790000e+02,2.147333e+09



Resumo das colunas categóricas:


,count,unique,top,freq
frame.time,157800,155186,192.168.0.128,1402
ip.src_host,157800,19090,192.168.0.128,72546
ip.dst_host,157800,8084,192.168.0.128,75373
arp.dst.proto_ipv4,157800,8,0,153610
arp.src.proto_ipv4,157800,8,0,140514
http.file_data,157800,496,0.0,117122
http.request.uri.query,157800,1665,0.0,137413
http.request.method,157800,6,0.0,96542
http.referer,157800,4,0.0,127111
http.request.full_uri,157800,4073,0.0,96542



Distribuição de Attack_label:


,quantidade,percentual (%)
Attack_label,,
1,133499,84.6
0,24301,15.4



Distribuição de Attack_type:


,quantidade,percentual (%)
Attack_type,,
Normal,24301,15.40
DDoS_UDP,14498,9.19
DDoS_ICMP,14090,8.93
Ransomware,10925,6.92
DDoS_HTTP,10561,6.69
SQL_injection,10311,6.53
Uploading,10269,6.51
DDoS_TCP,10247,6.49
Backdoor,10195,6.46


In [12]:
from time import perf_counter

# Attack_label is the binary target; exclude Attack_type to prevent target leakage.
# This baseline uses numeric predictors only; text/IP columns are left out.
model_df = df.drop_duplicates().copy()
y = model_df["Attack_label"].astype(int)
X = model_df.select_dtypes(include=np.number).drop(columns="Attack_label")

if X.empty:
    raise ValueError("No numeric predictor columns found after excluding Attack_label.")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Keep one shared stratified evaluation subset for a quick comparison.
X_test_eval, _, y_test_eval, _ = train_test_split(
    X_test,
    y_test,
    train_size=min(5_000, len(X_test)),
    random_state=42,
    stratify=y_test,
)

models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=300, class_weight="balanced")
    ),
    "Random Forest": make_pipeline(
        RandomForestClassifier(
            n_estimators=80,
            n_jobs=-1,
            class_weight="balanced_subsample",
            random_state=42,
        )
    ),
    "Hist Gradient Boosting": make_pipeline(
        HistGradientBoostingClassifier(max_iter=100, random_state=42)
    ),
    "KNN (10k train rows)": make_pipeline(
        StandardScaler(), KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
    ),
}

slow_model_train_limits = {"KNN (10k train rows)": 10_000}
results = []
predictions = {}

for name, model in models.items():
    train_limit = slow_model_train_limits.get(name)
    if train_limit is not None:
        X_fit, _, y_fit, _ = train_test_split(
            X_train,
            y_train,
            train_size=min(train_limit, len(X_train)),
            random_state=42,
            stratify=y_train,
        )
    else:
        X_fit, y_fit = X_train, y_train

    started = perf_counter()
    model.fit(X_fit, y_fit)
    y_pred = model.predict(X_test_eval)
    elapsed = perf_counter() - started
    predictions[name] = y_pred

    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test_eval)[:, 1]
    else:
        y_score = model.decision_function(X_test_eval)

    results.append({
        "model": name,
        "train_rows": len(X_fit),
        "test_rows": len(X_test_eval),
        "numeric_features": X.shape[1],
        "fit_predict_seconds": elapsed,
        "accuracy": accuracy_score(y_test_eval, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test_eval, y_pred),
        "precision_attack": precision_score(y_test_eval, y_pred, zero_division=0),
        "recall_attack": recall_score(y_test_eval, y_pred, zero_division=0),
        "f1_attack": f1_score(y_test_eval, y_pred, zero_division=0),
        "macro_f1": f1_score(y_test_eval, y_pred, average="macro", zero_division=0),
        "mcc": matthews_corrcoef(y_test_eval, y_pred),
        "cohen_kappa": cohen_kappa_score(y_test_eval, y_pred),
        "roc_auc": roc_auc_score(y_test_eval, y_score),
    })

results_df = pd.DataFrame(results).sort_values("macro_f1", ascending=False)
print(f"Rows after removing duplicates: {len(model_df):,}")
print(f"Training rows available: {len(X_train):,}; evaluation rows: {len(X_test_eval):,}")
display(results_df.round(4))

best_model_name = results_df.iloc[0]["model"]
best_predictions = predictions[best_model_name]
print(f"\nClassification report: {best_model_name}")
print(classification_report(y_test_eval, best_predictions, digits=4))
print("Confusion matrix: rows=actual, columns=predicted")
display(pd.DataFrame(
    confusion_matrix(y_test_eval, best_predictions),
    index=["actual 0 (benign)", "actual 1 (attack)"],
    columns=["predicted 0", "predicted 1"],
))

Rows after removing duplicates: 156,986
Training rows available: 125,588; evaluation rows: 5,000


,model,train_rows,test_rows,numeric_features,fit_predict_seconds,accuracy,balanced_accuracy,precision_attack,recall_attack,f1_attack,macro_f1,mcc,cohen_kappa,roc_auc
2,Hist Gradient Boosting,125588,5000,42,1.2698,0.9962,0.9909,0.9969,0.9986,0.9978,0.9927,0.9854,0.9854,0.9998
1,Random Forest,125588,5000,42,0.8937,0.9950,0.9881,0.9960,0.9981,0.9970,0.9904,0.9808,0.9808,0.9991
3,KNN (10k train rows),10000,5000,42,0.0927,0.9188,0.8042,0.9361,0.9702,0.9528,0.8308,0.6677,0.6622,0.9379
0,Logistic Regression,125588,5000,42,0.5240,0.7326,0.8117,0.9810,0.6971,0.8151,0.6663,0.4605,0.3789,0.8840



Classification report: Hist Gradient Boosting
              precision    recall  f1-score   support

           0     0.9922    0.9832    0.9877       774
           1     0.9969    0.9986    0.9978      4226

    accuracy                         0.9962      5000
   macro avg     0.9946    0.9909    0.9927      5000
weighted avg     0.9962    0.9962    0.9962      5000

Confusion matrix: rows=actual, columns=predicted


,predicted 0,predicted 1
actual 0 (benign),761,13
actual 1 (attack),6,4220


In [13]:
from time import perf_counter
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Attack_type reveals the subtype and is excluded to avoid target leakage.
# Only numeric predictors are used; categorical/IP/payload columns are not encoded here.
nn_df = df.drop_duplicates().copy()
X_nn = nn_df.drop(columns=["Attack_label", "Attack_type"]).select_dtypes(include=np.number)
y_nn = nn_df["Attack_label"].astype(np.int64)

X_train_nn, X_test_nn, y_train_nn, y_test_nn = train_test_split(
    X_nn, y_nn, test_size=0.2, random_state=42, stratify=y_nn
)
X_test_eval_nn, _, y_test_eval_nn, _ = train_test_split(
    X_test_nn,
    y_test_nn,
    train_size=min(5_000, len(X_test_nn)),
    random_state=42,
    stratify=y_test_nn,
)

# Fit scaling only on training data, matching the sklearn pipeline split.
scaler_nn = StandardScaler()
X_train_scaled = scaler_nn.fit_transform(X_train_nn).astype(np.float32)
X_test_scaled = scaler_nn.transform(X_test_eval_nn).astype(np.float32)
y_train_array = y_train_nn.to_numpy(dtype=np.float32).reshape(-1, 1)

train_dataset = TensorDataset(
    torch.from_numpy(X_train_scaled),
    torch.from_numpy(y_train_array),
)
train_loader = DataLoader(
    train_dataset, batch_size=1_024, shuffle=True, num_workers=0
)

nn_model = nn.Sequential(
    nn.Linear(X_train_scaled.shape[1], 64),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 1),
).to(device)

# Give benign and attack rows equal total influence during training.
class_counts_nn = np.bincount(y_train_nn.to_numpy(), minlength=2)
class_weights_nn = torch.tensor(
    len(y_train_nn) / (2.0 * class_counts_nn),
    dtype=torch.float32,
    device=device,
)
loss_fn = nn.BCEWithLogitsLoss(reduction="none")
optimizer = torch.optim.Adam(nn_model.parameters(), lr=1e-3, weight_decay=1e-4)

epochs = 15
started = perf_counter()
for epoch in range(epochs):
    nn_model.train()
    total_loss = 0.0
    for batch_X, batch_y in train_loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = nn_model(batch_X)
        row_losses = loss_fn(logits, batch_y)
        row_weights = class_weights_nn[batch_y.long()]
        loss = (row_losses * row_weights).mean()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(batch_X)

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1}/{epochs} - weighted loss: {total_loss / len(train_dataset):.4f}")

nn_model.eval()
with torch.no_grad():
    test_X_tensor = torch.from_numpy(X_test_scaled).to(device)
    y_score_nn = torch.sigmoid(nn_model(test_X_tensor)).cpu().numpy().ravel()
nn_seconds = perf_counter() - started
y_pred_nn = (y_score_nn >= 0.5).astype(int)

nn_metrics = {
    "model": "PyTorch MLP",
    "train_rows": len(X_train_nn),
    "test_rows": len(y_test_eval_nn),
    "numeric_features": X_nn.shape[1],
    "fit_predict_seconds": nn_seconds,
    "accuracy": accuracy_score(y_test_eval_nn, y_pred_nn),
    "balanced_accuracy": balanced_accuracy_score(y_test_eval_nn, y_pred_nn),
    "precision": precision_score(y_test_eval_nn, y_pred_nn, zero_division=0),
    "recall": recall_score(y_test_eval_nn, y_pred_nn, zero_division=0),
    "f1": f1_score(y_test_eval_nn, y_pred_nn, zero_division=0),
    "macro_f1": f1_score(y_test_eval_nn, y_pred_nn, average="macro", zero_division=0),
    "mcc": matthews_corrcoef(y_test_eval_nn, y_pred_nn),
    "cohen_kappa": cohen_kappa_score(y_test_eval_nn, y_pred_nn),
    "roc_auc": roc_auc_score(y_test_eval_nn, y_score_nn),
}
nn_metrics_df = pd.DataFrame([nn_metrics])
print(f"Device: {device}; rows after deduplication: {len(nn_df):,}; numeric features: {X_nn.shape[1]}")
display(nn_metrics_df.round(4))

print("Classification report - PyTorch MLP")
print(classification_report(
    y_test_eval_nn,
    y_pred_nn,
    labels=[0, 1],
    target_names=["benign", "attack"],
    digits=4,
))
print("Confusion matrix - rows=actual, columns=predicted")
display(pd.DataFrame(
    confusion_matrix(y_test_eval_nn, y_pred_nn, labels=[0, 1]),
    index=["actual 0 (benign)", "actual 1 (attack)"],
    columns=["predicted 0", "predicted 1"],
))

Epoch 5/15 - weighted loss: 0.2785
Epoch 10/15 - weighted loss: 0.1992
Epoch 15/15 - weighted loss: 0.1550
Device: cpu; rows after deduplication: 156,986; numeric features: 42


,model,train_rows,test_rows,numeric_features,fit_predict_seconds,accuracy,balanced_accuracy,precision,recall,f1,macro_f1,mcc,cohen_kappa,roc_auc
0,PyTorch MLP,125588,5000,42,11.5454,0.9332,0.9447,0.9924,0.9281,0.9592,0.8879,0.7903,0.777,0.9864


Classification report - PyTorch MLP
              precision    recall  f1-score   support

      benign     0.7099    0.9612    0.8167       774
      attack     0.9924    0.9281    0.9592      4226

    accuracy                         0.9332      5000
   macro avg     0.8512    0.9447    0.8879      5000
weighted avg     0.9487    0.9332    0.9371      5000

Confusion matrix - rows=actual, columns=predicted


,predicted 0,predicted 1
actual 0 (benign),744,30
actual 1 (attack),304,3922
